In [1]:
!pip -q install fastapi uvicorn python-multipart pymupdf requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 48.8 MB/s eta 0:00:00


In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


In [3]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 49 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (714 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
!ollama --version

In [6]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(8)

response = requests.get("http://127.0.0.1:11434")
print("Status code:", response.status_code)

Status code: 200


In [7]:
!ollama list

NAME    ID    SIZE    MODIFIED 


In [9]:
!ollama pull llama3

In [10]:
!ollama list

NAME             ID              SIZE      MODIFIED       
llama3:latest    365c0bd3c000    4.7 GB    44 seconds ago    


In [12]:
import requests

response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "llama3",
        "prompt": "Summarize in one sentence: Artificial intelligence helps computers perform tasks that normally require human intelligence.",
        "stream": False
    },
    timeout=300
)

print(response.json()["response"])

Artificial intelligence enables computers to perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making, by mimicking human thought processes and behaviors.


In [13]:
%%writefile main.py

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import HTMLResponse
import fitz
import requests

app = FastAPI()

OLLAMA_URL = "http://127.0.0.1:11434/api/generate"


def split_text(text, chunk_size=5000):
    return [
        text[i:i + chunk_size]
        for i in range(0, len(text), chunk_size)
    ]


def ask_llama(prompt):
    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": "llama3",
                "prompt": prompt,
                "stream": False
            },
            timeout=300
        )
        response.raise_for_status()
        return response.json()["response"]
    except requests.RequestException:
        raise HTTPException(
            status_code=503,
            detail="AI model is unavailable. Please try again."
        )


@app.get("/", response_class=HTMLResponse)
def home():
    return """
    <html>
    <head>
        <title>PDF Summarization Agent</title>
    </head>
    <body>
        <h2>PDF Summarization Agent</h2>
        <form action="/summarize" method="post"
              enctype="multipart/form-data">
            <input type="file" name="file" accept=".pdf"
                   required>
            <button type="submit">Summarize PDF</button>
        </form>
    </body>
    </html>
    """


@app.post("/summarize")
async def summarize_pdf(file: UploadFile = File(...)):
    if not file.filename.lower().endswith(".pdf"):
        raise HTTPException(
            status_code=400,
            detail="Please upload a PDF file."
        )

    content = await file.read()

    if len(content) > 10 * 1024 * 1024:
        raise HTTPException(
            status_code=400,
            detail="PDF size must be 10 MB or less."
        )

    if not content.startswith(b"%PDF-"):
        raise HTTPException(
            status_code=400,
            detail="Invalid or damaged PDF file."
        )

    try:
        with fitz.open(stream=content, filetype="pdf") as document:
            page_count = document.page_count
            text = "\n".join(page.get_text() for page in document)
    except Exception:
        raise HTTPException(
            status_code=400,
            detail="Unable to read this PDF."
        )

    if not text.strip():
        raise HTTPException(
            status_code=400,
            detail="No readable text found. Scanned PDFs may need OCR."
        )

    chunks = split_text(text)
    summaries = []

    for chunk in chunks:
        summary = ask_llama(
            "Summarize the following document section clearly. "
            "Keep the important information:\n\n" + chunk
        )
        summaries.append(summary)

    final_summary = ask_llama(
        "Create one clear, concise summary from these section summaries:\n\n"
        + "\n\n".join(summaries)
    )

    return {
        "filename": file.filename,
        "pages": page_count,
        "summary": final_summary
    }

Writing main.py


In [14]:
import threading
import time
import requests
import uvicorn
from main import app

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

api_thread = threading.Thread(target=run_server, daemon=True)
api_thread.start()

time.sleep(3)

response = requests.get("http://127.0.0.1:8000/")
print("Status code:", response.status_code)

INFO:     Started server process [3217]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:55880 - "GET / HTTP/1.1" 200 OK
Status code: 200


In [15]:
from google.colab import files
uploaded = files.upload()

Saving KRCE - HCL GUVI - Hackathon - Problem Statements.pdf to KRCE - HCL GUVI - Hackathon - Problem Statements.pdf


In [16]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
!./cloudflared tunnel --url http://127.0.0.1:8000 --no-autoupdate

2026-10-14T20:03:21Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-14T20:03:21Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-14T20:03:25Z INF +--------------------------------------------------------------------------------------------+
2026-10-14T20:03:25Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-14T20:03:25Z INF |  https://dressing-mortgage-spy-alternate.trycloudflare